# QPSO 2d

Reusable function definitions only. Original project inputs, run settings and saved outputs are not included. Running these cells defines functions; it does not read a dataset. Start with the [synthetic walkthrough](../examples/synthetic_walkthrough.ipynb) for a complete small run.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "helper" / "pipeline_utils.py").is_file()
             and (p / "examples" / "synthetic_demo.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start the notebook inside this repository.")
for folder in (ROOT, ROOT / "Invert-PropOnly-saveCSV", ROOT / "Invert-Prop+cond-saveCSV"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))


In [ ]:
import numpy as np
from invert_two_utils import (
    initialize_two_output_inversion,
    fitness_two_outputs,
    unpack_two_output_solutions,
    select_top_k_unique
)

def invert_two_outputs_qpso_Q(
    x_scaler_path, y_scaler_path,
    model1_path, model2_path,
    x_fixed_raw, actual_x_adj_raw,
    target1_raw, target2_raw,
    pop_size, num_iterations, beta,
    hidden_dims1, hidden_dims2,
    activation, x_adj_intervals_raw,
    Q_bounds, k
):
    # [1] Common init
    init = initialize_two_output_inversion(
        x_scaler_path, y_scaler_path,
        model1_path, model2_path,
        hidden_dims1, hidden_dims2,
        activation,
        x_fixed_raw, actual_x_adj_raw,
        target1_raw, target2_raw,
        x_adj_intervals_raw, Q_bounds
    )
    m1, m2             = init["model1"], init["model2"]
    means_x, scales_x  = init["means_x"], init["scales_x"]
    scales_y, mean_y   = init["scales_y"], init["mean_y"]
    x_fixed_s          = init["x_fixed_s"]
    results            = init["results"]
    min_vec, max_vec   = init["min_vec"], init["max_vec"]
    target_s           = init["target_s"]

    # [2] Init population
    population = [np.random.uniform(min_vec, max_vec) for _ in range(pop_size)]
    fitnesses  = [fitness_two_outputs(c, m1, m2, x_fixed_s, means_x, scales_x, target_s)
                  for c in population]
    pbest      = population.copy()
    pbest_f    = fitnesses.copy()

    # [3] QPSO loop
    for _ in range(num_iterations):
        mbest = np.mean(pbest, axis=0)
        for i, cand in enumerate(population):
            phi = np.random.rand(4)
            Pi  = phi*pbest[i] + (1-phi)*pbest[np.argmin(pbest_f)]

            u, sign = np.random.rand(4), np.where(np.random.rand(4)<0.5,1.0,-1.0)
            new_c   = Pi + sign*beta*np.abs(mbest-cand)*np.log(1.0/u)
            new_c   = np.clip(new_c, min_vec, max_vec)

            fval = fitness_two_outputs(new_c, m1, m2, x_fixed_s, means_x, scales_x, target_s)
            if fval < pbest_f[i]:
                pbest[i], pbest_f[i] = new_c.copy(), fval
            population[i] = new_c

    # [4] Select unique top-k
    best_scaled = select_top_k_unique(pbest_f, pbest, k)

    # [5] Inverse-scale & append
    solutions = unpack_two_output_solutions(
        best_scaled, m1, m2,
        means_x, scales_x,
        x_fixed_s,
        scales_y, mean_y,
        target1_raw, target2_raw
    )
    return results + solutions
